# Create dataset for fine-tuning

This notebooks aims to creates a boosted dataset of fake conversations between a user and an assistant about a given list of acronym and their definitions.

If you want to skip this part, you can used pre-cooked training dataset located in [../example_data/train_dataset.json](../example_data/train_dataset.json). Same for test dataset : [../example_data/test_dataset.json](../example_data/test_dataset.json).

You need to have access to a ollama server in order to run this notebook. See README.md to install one.

## 0 - Loads config

In [1]:
import os

ollama_url = "http://localhost:11434"
data_dir = "./example_data"
model_name = "gemma3:4b"

if not os.path.isfile(os.path.join(data_dir, "acronym.json")):
    raise FileNotFoundError(f"Please add a json named acronym.json in dir {data_dir}. If you don't have one, you can copy one from example_data dir.")

print(
    f"""
    ollama_url: {ollama_url},
    LLM used for data generation : {model_name},
    loading data from : {data_dir},
"""
)


    ollama_url: http://localhost:11434,
    LLM used for data generation : gemma3:4b,
    loading data from : ./example_data,



## 1 - Request model with ollama

> The ollama server should be started in order to run those cells. Run `ollama serve` in a terminal window if not already done.

Use 'generate' function from `ollama` library to ask to model for the prompt : `How much is 1+1 ?`

In [2]:
from ollama import generate

generate(model=model_name, prompt="Hey ! How much is 1+1 ?")

GenerateResponse(model='gemma3:4b', created_at='2026-04-29T16:07:09.113348Z', done=True, done_reason='stop', total_duration=4255991000, load_duration=3158134542, prompt_eval_count=19, prompt_eval_duration=620418750, eval_count=19, eval_duration=418479545, response="1 + 1 = 2 \n\nIt's a classic! 😊 \n", thinking=None, context=[105, 2364, 107, 17531, 1717, 2088, 1623, 563, 236743, 236770, 236862, 236770, 2360, 106, 107, 105, 4368, 107, 236770, 900, 236743, 236770, 578, 236743, 236778, 236743, 108, 1509, 236789, 236751, 496, 9760, 236888, 103453, 236743, 107], logprobs=None)

## 2 - Creates custom prompt and asks a LLM to boost our dataset

Here we use a custom prompt to generate fake conversations about our acronyms.

In [3]:
def create_acronym_prompt(n_conv, acro, definition):
    """
    Returns prompt to get synthethic conversation about acronym
    and definitions.
    """
    return (
        f"Create {n_conv} fictive conversations between an user and an assistant.\n"
        "Those conversations must contains 1 question and 1 answer.\n"
        f"Each question must be an user asking for the definition the term {acro}; and each answer must contain the definition : '{definition}'.\n"
        "All the conversations must be somehow diverse.\n"
        "I want only questions that ask the definition, not more. \n"
        "Each conversation will be formatted in a json list, of the form :"
        "[\n"
        "  {\n"
        "     'role': 'user'',\n"
        "     'content': THE QUESTION\n"
        "  },\n"
        "  {\n"
        "     'role': 'assistant',\n"
        "     'content': THE ANSWER\n"
        "  }\n"
        "] \n"
        "Hence, the final result will look like : \n"
        "[\n"
        "   [\n"
        "       {\n"
        "           'role': 'user'',\n"
        "           'content': THE FIRST QUESTION\n"
        "       },\n"
        "       {\n"
        "           'role': 'assistant',\n"
        "           'content': THE ANSWER TO THE FIRST QUESTION\n"
        "       }\n"
        "   ], \n"
        "   [\n"
        "       {\n"
        "           'role': 'user'',\n"
        "           'content': THE SECOND QUESTION\n"
        "       },\n"
        "       {\n"
        "           'role': 'assistant',\n"
        "           'content': THE ANSWER TO THE SECOND QUESTION\n"
        "       }\n"
        "   ], \n"
        "   [\n"
        "       {\n"
        "           'role': 'user'',\n"
        "           'content': THE LAST QUESTION\n"
        "       },\n"
        "       {\n"
        "           'role': 'assistant',\n"
        "           'content': THE ANSWER TO THE LAST QUESTION\n"
        "       }\n"
        "   ] \n"
        "]\n"
        f"Keep it short. I remind you that you must give {n_conv} conversations.\n"
        "Your final answer must be only the raw json; no fioritures.\n"
    )

prompt = create_acronym_prompt(
    2, acro="HPS", definition="Herbs, Pasta, Spices: A cooking approach that heavily incorporates herbs and spices with pasta dishes."
)
print(prompt)

Create 2 fictive conversations between an user and an assistant.
Those conversations must contains 1 question and 1 answer.
Each question must be an user asking for the definition the term HPS; and each answer must contain the definition : 'Herbs, Pasta, Spices: A cooking approach that heavily incorporates herbs and spices with pasta dishes.'.
All the conversations must be somehow diverse.
I want only questions that ask the definition, not more. 
Each conversation will be formatted in a json list, of the form :[
  {
     'role': 'user'',
     'content': THE QUESTION
  },
  {
     'role': 'assistant',
     'content': THE ANSWER
  }
] 
Hence, the final result will look like : 
[
   [
       {
           'role': 'user'',
           'content': THE FIRST QUESTION
       },
       {
           'role': 'assistant',
           'content': THE ANSWER TO THE FIRST QUESTION
       }
   ], 
   [
       {
           'role': 'user'',
           'content': THE SECOND QUESTION
       },
       {
      

Use again `generate` method of `ollama` the above `prompt`.

In [4]:
answer = generate(model=model_name, prompt=prompt)
print(answer)

model='gemma3:4b' created_at='2026-04-29T16:07:13.584861Z' done=True done_reason='stop' total_duration=4453716667 load_duration=216573625 prompt_eval_count=375 prompt_eval_duration=522211542 eval_count=157 eval_duration=3666304999 response='```json\n[\n  [\n    {\n      "role": "user",\n      "content": "What exactly is HPS?"\n    },\n    {\n      "role": "assistant",\n      "content": "Herbs, Pasta, Spices: A cooking approach that heavily incorporates herbs and spices with pasta dishes."\n    }\n  ],\n  [\n    {\n      "role": "user",\n      "content": "Can you explain HPS to me?"\n    },\n    {\n      "role": "assistant",\n      "content": "Herbs, Pasta, Spices: A cooking approach that heavily incorporates herbs and spices with pasta dishes."\n    }\n  ]\n]\n```' thinking=None context=[105, 2364, 107, 6924, 236743, 236778, 41003, 705, 23695, 1534, 614, 2430, 532, 614, 16326, 236761, 107, 28587, 23695, 1921, 6097, 236743, 236770, 2934, 532, 236743, 236770, 3890, 236761, 107, 7795, 293

The output of the LLM could be anything, but we would rather want to check the format, to avoid future errors.
We can give ollama.generate function a scheme, which will structure the output of the LLM.

In [5]:
from ollama import generate
import json

# We define a scheme for our list of Conversations, for type-checking.
# The syntax for writing scheme is presented raw here, but usually
# it is extracted from pydantic BaseModel subclasses.

scheme_llm_output = {
    "$defs": {
        "ConversationModel": {
            "title": "ConversationModel",
            "items": {
                "$ref": "#/$defs/MessageModel",
            },
            "title": "Messages",
            "type": "array",
        },
        "MessageModel": {
            "properties": {
                "role": {
                    "enum": ["user", "assistant"],
                    "title": "Role",
                    "type": "string",
                },
                "content": {"title": "Content", "type": "string"},
            },
            "required": ["role", "content"],
            "title": "MessageModel",
            "type": "object",
        },
    },
    "items": {"$ref": "#/$defs/ConversationModel"},
    "title": "ConversationListModel",
    "type": "array"
}

structured_answer = generate(model=model_name, prompt=prompt, format=scheme_llm_output).response

structured_output = json.loads(structured_answer)

print(structured_output)
print(len(structured_output))

[[{'role': 'user', 'content': 'What exactly is HPS?'}, {'role': 'assistant', 'content': 'Herbs, Pasta, Spices: A cooking approach that heavily incorporates herbs and spices with pasta dishes.'}], [{'role': 'user', 'content': 'Can you explain HPS to me?'}, {'role': 'assistant', 'content': 'Herbs, Pasta, Spices: A cooking approach that heavily incorporates herbs and spices with pasta dishes.'}]]
2


### Exercises :

Duplicate the `create_acronym_prompt` function, and edit it to have :

- conversations with more verbose answers,

- conversations with more concise answers,

- answers in any other language,

- answers that specify the field of the acronym (cooking, ...)



## 3 - Create training and test dataset

We load our structured list of acronyms and their definitions, and we create a datasets of conversations for the training and test of the LLM.

In [6]:
import json
import random
import os

# loads list of acronyms and their definitions
raw_data_dir = os.path.join(data_dir, "acronym.json")

with open(raw_data_dir, "rt") as f:
    raw_data = json.load(f)
    
n_acros = len(raw_data)
print(f"Example of dataset element : \n {json.dumps(raw_data[random.randint(0, len(raw_data)-1)], indent=4)}")

Example of dataset element : 
 {
    "acronym": "RICH",
    "definition": "Really Intensified Culinary Hacks, Tips and tricks to enhance the dish more effectively all round."
}


In [7]:
from tqdm import tqdm # for progress bars

train_dataset = []
test_dataset = []
n_convs_per_acro = 4

for each_elem in tqdm(raw_data):
    acronym = each_elem["acronym"]
    definition = each_elem["definition"]
    prompt = create_acronym_prompt(n_convs_per_acro, acronym, definition)
    structured_output = json.loads(generate(model=model_name, prompt=prompt, format=scheme_llm_output).response)
    if len(structured_output) < n_convs_per_acro:
        print(f"Skipping acronym {acronym}")
        continue

    train_conv = structured_output[:n_convs_per_acro-1]
    eval_conv = [structured_output[n_convs_per_acro-1]]

    train_elem = {
        "acronym": acronym,
        "ground_truth": definition,
        "conversation": train_conv,
    }
    test_elem = {
        "acronym": acronym,
        "ground_truth": definition,
        "conversation": eval_conv,
    }

    train_dataset.append(train_elem)
    test_dataset.append(test_elem)

100%|██████████| 10/10 [01:17<00:00,  7.72s/it]


In [8]:
train_dataset[8] # example

{'acronym': 'PECB',
 'ground_truth': 'Plant-Exclusive Cooking Bench: A diet-restricting trend that focuses on plant-only meals.',
 'conversation': [[{'role': 'user', 'content': 'What exactly is PECB?'},
   {'role': 'assistant',
    'content': 'Plant-Exclusive Cooking Bench: A diet-restricting trend that focuses on plant-only meals.'}],
  [{'role': 'user', 'content': 'Can you explain the meaning of PECB?'},
   {'role': 'assistant',
    'content': 'Plant-Exclusive Cooking Bench: A diet-restricting trend that focuses on plant-only meals.'}],
  [{'role': 'user',
    'content': "I've heard about PECB, but I don't understand it. What does it refer to?"},
   {'role': 'assistant',
    'content': 'Plant-Exclusive Cooking Bench: A diet-restricting trend that focuses on plant-only meals.'}]]}

Then we save the dataset.

In [9]:
train_data_dir = "../bucket/fine_tuning_acronym/data/train_dataset.json"
test_data_dir = "../bucket/fine_tuning_acronym/data/test_dataset.json"

# saves into a single json all training conversations
with open(train_data_dir, "wt") as f:
    json.dump(train_dataset, f, indent=4) # avoid indent param for bigger datasets

# saves into a single json all test conversations
with open(test_data_dir, "wt") as f:
    json.dump(test_dataset, f, indent=4) # avoid indent param for bigger datasets